# Optimizing Strategy Parameters

In [ ]:
???

<table style="width:100%; height:90%">
      <tr>
    <th>Parametrize the Strategy</th>
    <th>Optimizing Limits' Parameters</th>
  </tr>
  <tr>
    <td><img src="src/07_Code_Regression Strategy Limits X.png" alt="Parametrize the Strategy" style="width:100%"></td>
    <td><img src="src/07_Table_Optimize BG Default Defaults.png" alt="Optimizing Limits' Parameters" style="width:100%"></td>
  </tr>
</table>

## Load the model

In [1]:
import pickle

with open('models/model_dt_regression.pkl', 'rb') as f:
    model_dt = pickle.load(f)
    
model_dt

DecisionTreeRegressor(max_depth=15)

## Load the data

In [2]:
import pandas as pd

df = pd.read_excel('data/Microsoft_LinkedIn_Processed.xlsx', index_col=0, parse_dates=['Date'])
df

,Close,High,Low,Open,Volume,change_tomorrow,change_tomorrow_direction
Date,,,,,,,
2016-12-08,61.009998,61.580002,60.840000,61.299999,21220800,1.549141,UP
2016-12-09,61.970001,61.990002,61.130001,61.180000,27349400,0.321694,UP
2016-12-12,62.169998,62.299999,61.720001,61.820000,20198100,1.286125,UP
2016-12-13,62.980000,63.419998,62.240002,62.500000,35718900,-0.478620,DOWN
2016-12-14,62.680000,63.450001,62.529999,63.000000,30352700,-0.159793,DOWN
...,...,...,...,...,...,...,...
2026-05-14,409.429993,411.839996,400.880005,404.480011,27077500,2.960282,UP
2026-05-15,421.920013,428.170013,412.910004,414.269989,50771200,0.382489,UP
2026-05-18,423.540009,425.119995,415.609985,416.619995,32564100,-1.466148,DOWN


# Simple Investment Strategy

### Create Strategy class

In [3]:
from backtesting import Strategy, Backtest

In [4]:
class Regression(Strategy):
    def init(self):
        self.model = model_dt
        self.already_bought = False

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > 1 and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < -5 and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

### Create Backtest class

In [5]:
df_explanatory = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

In [6]:
bt = Backtest(df_explanatory, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

### Run backtesting with specific values

In [7]:
results = bt.run()

### Interpret backtesting results

In [8]:
results.to_frame(name='Values').loc[:'Return [%]']

,Values
Start,2016-12-08 00:00:00
End,2026-05-20 00:00:00
Duration,3450 days 00:00:00
Exposure Time [%],99.705139
Equity Final [$],117696.819509
Equity Peak [$],131431.224984
Return [%],1076.968195


## Parametrize the Investment Strategy

### Create Strategy class

In [9]:
from backtesting import Strategy, Backtest

In [10]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5
    
    def init(self):
        self.model = model_dt
        self.already_bought = False

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

### Create Backtest class

In [11]:
bt = Backtest(df_explanatory, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

## Optimize backtesting with multiple combinations

In [12]:
list_limits_buy = list(range(0, 11, 1))
list_limits_buy

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [13]:
list_limits_sell = list(range(0, -11, -1))
list_limits_sell

[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10]

In [14]:
%%time

results = bt.optimize(
    limit_buy = list_limits_buy, limit_sell = list_limits_sell,
    maximize='Return [%]', return_heatmap=True
)

/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/backtesting.py:1375: UserWarning: For multiprocessing support in `Backtest.optimize()` set multiprocessing start method to 'fork'.
  warnings.warn("For multiprocessing support in `Backtest.optimize()` "


CPU times: user 1min 47s, sys: 1.46 s, total: 1min 48s
Wall time: 1min 52s


In [16]:
results[0]

Start                     2016-12-08 00:00:00
End                       2026-05-20 00:00:00
Duration                   3450 days 00:00:00
Exposure Time [%]                   99.915754
Equity Final [$]               6211773.394972
Equity Peak [$]                6211773.394972
Return [%]                        62017.73395
Buy & Hold Return [%]              590.149171
Return (Ann.) [%]                   97.924574
Volatility (Ann.) [%]               53.801452
Sharpe Ratio                          1.82011
Sortino Ratio                        5.933841
Calmar Ratio                         4.787451
Max. Drawdown [%]                   -20.45443
Avg. Drawdown [%]                   -2.697694
Max. Drawdown Duration      182 days 00:00:00
Avg. Drawdown Duration       14 days 00:00:00
# Trades                                  544
Win Rate [%]                        63.786765
Best Trade [%]                      33.309739
Worst Trade [%]                    -13.404178
Avg. Trade [%]                    

In [17]:
results[1]

limit_buy  limit_sell
0           0            62017.733950
           -1            34666.024856
           -2             7950.344143
           -3             2255.654016
           -4             1554.264692
                             ...     
10         -6             -100.000000
           -7             -100.000000
           -8             -100.000000
           -9             -100.000000
           -10            -100.000000
Name: Return [%], Length: 121, dtype: float64

### [ ] Interpret optimization results

In [18]:
dff = results[1].reset_index()
dff

,limit_buy,limit_sell,Return [%]
0,0,0,62017.733950
1,0,-1,34666.024856
2,0,-2,7950.344143
3,0,-3,2255.654016
4,0,-4,1554.264692
...,...,...,...
116,10,-6,-100.000000
117,10,-7,-100.000000
118,10,-8,-100.000000
119,10,-9,-100.000000


In [21]:
print(dff.columns.tolist())

['limit_buy', 'limit_sell', 'Return [%]']


In [24]:
dff= dff.pivot(index='limit_buy', columns='limit_sell', values='Return [%]')

### DataFrame heatmaps for better reporting

In [23]:
dff.sort_index(axis=1, ascending=False)\
  .style.format(precision=0)\
    .background_gradient(vmin=dff.values.min(), vmax=dff.values.max())

,limit_sell,limit_buy,Return [%]
0,0,0,62018
1,-1,0,34666
2,-2,0,7950
3,-3,0,2256
4,-4,0,1554
5,-5,0,1076
6,-6,0,862
7,-7,0,682
8,-8,0,658
9,-9,0,630
